In [0]:
%run ../helper/silver_helper

In [0]:

bronze_table = f"{catalog}.bronze.hacker_news_historical"

checkpoint_root = (
    "/Volumes/job_market/landing/raw_files/"
    "_pipeline_state/"
)

# Bronze inkrementell lesen
df = spark.readStream.table(bronze_table)

# Transformationen
df = clean_hn(df)

# Gültige und ungültige Daten trennen
valid = df.filter(
    ~F.col("is_quarantined")
).dropDuplicates(["post_id"])

invalid = df.filter(
    F.col("is_quarantined")
)


In [0]:

query = (
    valid.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_root + "hn_silver_valid_v1"
    )
    .trigger(availableNow=True)
    .toTable("job_market.silver.hacker_news_posts")
)

query.awaitTermination()


In [0]:

query = (
    invalid.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_root + "hn_silver_quarantine_v1"
    )
    .trigger(availableNow=True)
    .toTable("job_market.silver.hacker_news_quarantine")
)

query.awaitTermination()


In [0]:
%sql
SELECT
    s.post_id,
    s.posted_year,
    d.post_id IS NOT NULL AS has_duplicates
FROM job_market.silver.hacker_news_posts s
LEFT JOIN job_market.silver.hn_duplicate_audit d
    ON s.post_id = d.post_id;

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT post_id) AS unique_posts,
    COUNT_IF(is_quarantined) AS quarantined_rows
FROM job_market.silver.hacker_news_posts;


In [0]:
%sql

SELECT
    COUNT(*) AS duplicated_post_ids,
    COALESCE(SUM(occurrences - 1), 0)
        AS extra_duplicate_rows
FROM job_market.silver.hn_duplicate_audit;


In [0]:
%sql

SELECT
    quarantine_reason,
    COUNT(*) AS total
FROM job_market.silver.hacker_news_quarantine
GROUP BY quarantine_reason;
